In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Merge_bases") \
    .master("local[*]") \
    .getOrCreate()

print("Spark pret. Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 00:12:54 WARN Utils: Your hostname, codespaces-2898b9, resolves to a loopback address: 127.0.0.1; using 10.0.0.212 instead (on interface eth0)
26/04/18 00:12:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 00:12:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/18 00:12:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark pret. Version : 4.1.1


In [2]:
etreproprio = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("inferSchema", True) \
    .csv("etreproprio_rhone_propre.csv")

logimmo = spark.read.csv("logic_immo_lyon_propre.csv", header=True, inferSchema=True, sep=";")


print("EP, Nombre de lignes lues :", etreproprio.count())
etreproprio.show(5, truncate=False)

print("LI, Nombre de lignes lues :", logimmo.count())
logimmo.show(5, truncate=False)



EP, Nombre de lignes lues : 578
+-----------------------------------------------+----------------------------------------------------------------------------------------------+----------------------+-----------+-----+--------+-------+------+---+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
etreproprio = etreproprio.withColumn(
    "ville",
    F.lower(
        F.trim(
            F.translate(
                F.col("ville"),
                "àâäéèêëîïôöùûüç-",
                "aaaeeeeiioouuuc ")
            )
        )
    )

logimmo = logimmo.withColumn(
    "ville",
    F.lower(
        F.trim(
            F.translate(
                F.col("ville"),
                "àâäéèêëîïôöùûüç-",
                "aaaeeeeiioouuuc ")
            )
        )
    )

In [4]:
etreproprio = etreproprio.withColumn(
    "ville",
    F.trim(
        F.regexp_replace(
            F.col("ville"),
            r"\s*\d+e\b", "")
        )
    )

In [5]:
a = etreproprio.select("ville").distinct()
b = logimmo.select("ville").distinct()

In [6]:
a.join(b, a.ville == b.ville, "left_anti").show(truncate=False,n=200)

+--------------------------+
|ville                     |
+--------------------------+
|grigny                    |
|saint maurice sur dargoire|
|echalas                   |
|saint clement sur valsonne|
|collonges au mont d'or    |
|chenas                    |
|marcy l'etoile            |
|belleville                |
|corcelles en beaujolais   |
|oullins                   |
|liergues                  |
|le bois d'oingt           |
|oingt                     |
|pierre benite             |
|ecully                    |
|sainte foy les lyon       |
+--------------------------+



In [7]:
b.show(truncate=False,n=5000)

+---------------------------------+
|ville                            |
+---------------------------------+
|poleymieux au mont d'or          |
|chaponnay                        |
|lancie                           |
|sourcieux les mines              |
|vaugneray                        |
|marcilly d'azergues              |
|montromant                       |
|saint didier au mont d'or        |
|brindas                          |
|solaize                          |
|meys                             |
|rivolet                          |
|sain bel                         |
|les cheres                       |
|saint forgeux                    |
|feyzin                           |
|val d'oingt                      |
|civrieux d'azergues              |
|amplepuis                        |
|albigny sur saone                |
|les ardillats                    |
|ternand                          |
|saint andre la cote              |
|ampuis                           |
|saint clement de vers      

In [8]:
a.join(b, a.ville == b.ville, "inner").show(truncate=False,n=200)

+-------------------------+-------------------------+
|ville                    |ville                    |
+-------------------------+-------------------------+
|poleymieux au mont d'or  |poleymieux au mont d'or  |
|chaponnay                |chaponnay                |
|sourcieux les mines      |sourcieux les mines      |
|marcilly d'azergues      |marcilly d'azergues      |
|saint didier au mont d'or|saint didier au mont d'or|
|rivolet                  |rivolet                  |
|sain bel                 |sain bel                 |
|feyzin                   |feyzin                   |
|amplepuis                |amplepuis                |
|albigny sur saone        |albigny sur saone        |
|saint clement de vers    |saint clement de vers    |
|saint priest             |saint priest             |
|fontaines sur saone      |fontaines sur saone      |
|pollionnay               |pollionnay               |
|dardilly                 |dardilly                 |
|tarare                   |t

In [9]:
etreproprio = etreproprio.withColumn(
    "type_bien",
    F.lower("type_bien")
    ).withColumn(
    "ville",
    F.when(F.col("ville")=="grigny", "grigny sur rhone"
            ).when(F.col("ville")=="saint maurice sur dargoire","chabaniere"
            ).when(F.col("ville")=="belleville","belleville en beaujolais"
            ).when((F.col("ville")=="oullins") | (F.col("ville")=="pierre benite"),"oullins pierre benite"
            ).when(F.col("ville")=="liergues","porte des pierres dorees"
            ).when((F.col("ville")=="le bois d'oingt") | (F.col("ville")=="oingt"), "val d'oingt"
            ).otherwise(F.col("ville"))
    ).withColumn("etage",
        F.when(F.col("type_bien")=="maison","MAISON"
                ).when(F.col("etage")==0, "RDC"
                ).otherwise(F.col("etage").cast("string")))
logimmo = logimmo.withColumnRenamed("type_clean", "type_bien") \
       .withColumnRenamed("etage_clean", "etage")

In [10]:
#liste des communes du rhones
liste_communes_brute = """Affoux (69001)
Aigueperse (69002)
Albigny-sur-Saône (69003)
Alix (69004)
Ambérieux (69005)
Amplepuis (69006)
Ampuis (69007)
Ancy (69008)
Anse (69009)
L'Arbresle (69010)
Les Ardillats (69012)
Arnas (69013)
Aveize (69014)
Azolette (69016)
Bagnols (69017)
Beaujeu (69018)
Beauvallon (69179)
Belleville-en-Beaujolais (69019)
Belmont-d'Azergues (69020)
Bessenay (69021)
Bibost (69022)
Blacé (69023)
Le Breuil (69026)
Brignais (69027)
Brindas (69028)
Bron (69029)
Brullioles (69030)
Brussieu (69031)
Bully (69032)
Cailloux-sur-Fontaines (69033)
Caluire-et-Cuire (69034)
Cenves (69035)
Cercié (69036)
Chabanière (69228)
Chambost-Allières (69037)
Chambost-Longessaigne (69038)
Chamelet (69039)
Champagne-au-Mont-d'Or (69040)
La Chapelle-sur-Coise (69042)
Chaponnay (69270)
Chaponost (69043)
Charbonnières-les-Bains (69044)
Charentay (69045)
Charly (69046)
Charnay (69047)
Chasselay (69049)
Chassieu (69271)
Châtillon (69050)
Chaussan (69051)
Chazay-d'Azergues (69052)
Chénas (69053)
Chénelette (69054)
Les Chères (69055)
Chessy (69056)
Chevinay (69057)
Chiroubles (69058)
Civrieux-d'Azergues (69059)
Claveisolles (69060)
Cogny (69061)
Coise (69062)
Collonges-au-Mont-d'Or (69063)
Colombier-Saugnieu (69299)
Communay (69272)
Condrieu (69064)
Corbas (69273)
Corcelles-en-Beaujolais (69065)
Cours (69066)
Courzieu (69067)
Couzon-au-Mont-d'Or (69068)
Craponne (69069)
Cublize (69070)
Curis-au-Mont-d'Or (69071)
Dardilly (69072)
Décines-Charpieu (69275)
Denicé (69074)
Deux-Grosnes (69135)
Dième (69075)
Dommartin (69076)
Dracé (69077)
Duerne (69078)
Échalas (69080)
Écully (69081)
Émeringes (69082)
Éveux (69083)
Feyzin (69276)
Fleurie (69084)
Fleurie-sur-Saône (69085)
Fleurieux-sur-l'Arbresle (69086)
Fontaines-Saint-Martin (69087)
Fontaines-sur-Saône (69088)
Francheville (69089)
Frontenas (69090)
Genas (69277)
Genay (69278)
Givors (69091)
Gleizé (69092)
Grandris (69093)
Grézieu-la-Varenne (69094)
Grézieu-le-Marché (69095)
Grigny-sur-Rhône (69096)
Les Haies (69097)
Les Halles (69098)
Haute-Rivoire (69099)
Irigny (69100)
Jonage (69279)
Jons (69280)
Joux (69102)
Juliénas (69103)
Jullié (69104)
Lacenas (69105)
Lachassagne (69106)
Lamure-sur-Azergues (69107)
Lancié (69108)
Lantignié (69109)
Larajasse (69110)
Légny (69111)
Lentilly (69112)
Létra (69113)
Limas (69115)
Limonest (69116)
Lissieu (69117)
Loire-sur-Rhône (69118)
Longes (69119)
Longessaigne (69120)
Lozanne (69121)
Lucenay (69122)
Lyon (69123)
Marchampt (69124)
Marcilly-d'Azergues (69125)
Marcy (69126)
Marcy-l'Étoile (69127)
Marennes (69281)
Meaux-la-Montagne (69130)
Messimy (69131)
Meys (69132)
Meyzieu (69282)
Millery (69133)
Mions (69283)
Moiré (69134)
Montagny (69136)
Montanay (69284)
Montmelas-Saint-Sorlin (69137)
Montromant (69138)
Montrottier (69139)
Morancé (69140)
Mornant (69141)
La Mulatière (69142)
Neuville-sur-Saône (69143)
Odenas (69145)
Orliénas (69148)
Oullins-Pierre-Bénite (69149)
Le Perréon (69151)
Poleymieux-au-Mont-d'Or (69153)
Pollionnay (69154)
Pomeys (69155)
Pommiers (69156)
Porte des Pierres Dorées (69114)
Poule-les-Écharmeaux (69160)
Propières (69161)
Pusignan (69285)
Quincié-en-Beaujolais (69162)
Quincieux (69163)
Ranchal (69164)
Régnié-Durette (69165)
Rillieux-la-Pape (69286)
Riverie (69166)
Rivolet (69167)
Rochetaillée-sur-Saône (69168)
Ronno (69169)
Rontalon (69170)
Sain-Bel (69171)
Saint-André-la-Côte (69180)
Saint-Appolinaire (69181)
Saint-Bonnet-de-Mure (69287)
Saint-Bonnet-des-Bruyères (69182)
Saint-Bonnet-le-Troncy (69183)
Saint-Clément-de-Vers (69186)
Saint-Clément-les-Places (69187)
Saint-Clément-sur-Valsonne (69188)
Saint-Cyr-au-Mont-d'Or (69191)
Saint-Cyr-le-Chatoux (69192)
Saint-Cyr-sur-le-Rhône (69193)
Saint-Didier-au-Mont-d'Or (69194)
Saint-Didier-sur-Beaujeu (69196)
Saint-Étienne-des-Oullières (69197)
Saint-Étienne-la-Varenne (69198)
Saint-Fons (69199)
Saint-Forgeux (69200)
Saint-Genis-l'Argentière (69203)
Saint-Genis-Laval (69204)
Saint-Genis-les-Ollières (69205)
Saint-Georges-de-Reneins (69206)
Saint-Germain-au-Mont-d'Or (69207)
Saint-Germain-Nuelles (69208)
Saint-Igny-de-Vers (69209)
Saint-Jean-des-Vignes (69212)
Saint-Jean-la-Bussière (69214)
Saint-Julien (69215)
Saint-Julien-sur-Bibost (69216)
Saint-Just-d'Avray (69217)
Saint-Lager (69218)
Saint-Laurent-d'Agny (69219)
Saint-Laurent-de-Chamousset (69220)
Saint-Laurent-de-Mure (69288)
Saint-Marcel-l'Éclairé (69225)
Saint-Martin-en-Haut (69227)
Saint-Nizier-d'Azergues (69229)
Saint-Pierre-de-Chandieu (69289)
Saint-Pierre-la-Palud (69231)
Saint-Priest (69290)
Saint-Romain-au-Mont-d'Or (69233)
Saint-Romain-de-Popey (69234)
Saint-Romain-en-Gal (69235)
Saint-Romain-en-Gier (69236)
Saint-Symphorien-d'Ozon (69291)
Saint-Symphorien-sur-Coise (69238)
Saint-Vérand (69239)
Saint-Vincent-de-Reins (69240)
Sainte-Catherine (69184)
Sainte-Colombe (69189)
Sainte-Consorce (69190)
Sainte-Foy-l'Argentière (69201)
Sainte-Foy-lès-Lyon (69202)
Sainte-Paule (69230)
Salles-Arbuissonnas-en-Beaujolais (69172)
Sarcey (69173)
Sathonay-Camp (69292)
Sathonay-Village (69293)
Les Sauvages (69174)
Savigny (69175)
Sérézin-du-Rhône (69294)
Simandres (69295)
Solaize (69296)
Soucieu-en-Jarrest (69176)
Sourcieux-les-Mines (69177)
Souzy (69178)
Taluyers (69241)
Taponas (69242)
Tarare (69243)
Tassin-la-Demi-Lune (69244)
Ternand (69245)
Ternay (69297)
Theizé (69246)
Thizy-les-Bourgs (69248)
Thurins (69249)
La Tour-de-Salvagny (69250)
Toussieu (69298)
Trèves (69252)
Tupin-et-Semons (69253)
Val d'Oingt (69024)
Valsonne (69254)
Vaugneray (69255)
Vaulx-en-Velin (69256)
Vaux-en-Beaujolais (69257)
Vauxrenard (69258)
Vénissieux (69259)
Vernaison (69260)
Vernay (69261)
Ville-sur-Jarnioux (69265)
Villechenève (69263)
Villefranche-sur-Saône (69264)
Villeurbanne (69266)
Villié-Morgon (69267)
Vindry-sur-Turdine (69157)
Vourles (69268)
Yzeron (69269)"""


In [11]:
city = spark.createDataFrame(
    [(x,) for x in liste_communes_brute.split("\n")],
    ["ville_brute"]
)

In [12]:
city = city.withColumn("ville_brute", F.trim(F.lower("ville_brute"))) \
       .withColumn("ville_og", F.regexp_extract("ville_brute", r"(.*)\s\((\d+)\)", 1)) \
       .withColumn("ville", F.lower(F.trim(F.translate(F.col("ville_og"),"àâäéèêëîïôöùûüç-","aaaeeeeiioouuuc ")))) \
       .withColumn("code_postal", F.regexp_extract("ville_brute", r"(.*)\s\((\d+)\)", 2))

In [13]:
etreproprio = etreproprio.join(city.select("ville", "code_postal"), on="ville", how="left")

In [14]:
etreproprio.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in etreproprio.columns
]).show()

+-----+-----+-----------+---------+-----+----+-------+------+---+----------------+----+-----------+
|ville|titre|description|type_bien|etage|prix|surface|pieces|DPE|full_description|lien|code_postal|
+-----+-----+-----------+---------+-----+----+-------+------+---+----------------+----+-----------+
|    0|    0|          0|        0|    0|   0|      0|     0|  0|               0|   0|          0|
+-----+-----+-----------+---------+-----+----+-------+------+---+----------------+----+-----------+



In [15]:
etreproprio.show(5)
logimmo.show(5, truncate=False)

+--------------------+--------------------+--------------------+-----------+------+--------+-------+------+---+--------------------+--------------------+-----------+
|               ville|               titre|         description|  type_bien| etage|    prix|surface|pieces|DPE|    full_description|                lien|code_postal|
+--------------------+--------------------+--------------------+-----------+------+--------+-------+------+---+--------------------+--------------------+-----------+
|              feyzin|Vente Appartement...|EXCLUSIVITÉ   Fey...|appartement|   RDC|380000.0|  208.0|   5.0|  A|FEYZIN LA BÉGUDE ...|https://www.etrep...|      69276|
|                lyon|Vente Appartement...|Foch / Malesherbe...|appartement|     1|645000.0|  152.0|   5.0|  E|Quartier Foch, id...|https://www.etrep...|      69123|
|        francheville|Vente Appartement...|Appartement 72 m2...|appartement|    -1|269000.0|   72.0|   3.0|  D|Decultieux Immobi...|https://www.etrep...|      69089|
|   

In [16]:
fusion_full=logimmo.unionByName(etreproprio, allowMissingColumns=True)
fusion_full.show(5, truncate=False)

+-----------+------+--------+--------+---------+-------+-------+-----+------------+-----------+-----+-----------+----+----------------+----+
|type_bien  |pieces|chambres|prix    |neuf_flag|surface|terrain|etage|ville       |code_postal|titre|description|DPE |full_description|lien|
+-----------+------+--------+--------+---------+-------+-------+-----+------------+-----------+-----+-----------+----+----------------+----+
|appartement|2.0   |1       |203431.0|1        |21.0   |0.0    |1    |venissieux  |69200      |NULL |NULL       |NULL|NULL            |NULL|
|appartement|2.0   |1       |235000.0|0        |38.9   |0.0    |6    |lyon        |69007      |NULL |NULL       |NULL|NULL            |NULL|
|appartement|2.0   |1       |454000.0|1        |92.0   |0.0    |1    |toussieu    |69780      |NULL |NULL       |NULL|NULL            |NULL|
|appartement|2.0   |1       |118000.0|0        |36.5   |0.0    |1    |saint priest|69800      |NULL |NULL       |NULL|NULL            |NULL|
|appartement|

In [17]:
common_cols = [c for c in logimmo.columns if c in etreproprio.columns]
df_union = logimmo.select(common_cols).unionByName(
    etreproprio.select(common_cols)
)
df_union.show(5, truncate=False)

+-----------+------+--------+-------+-----+------------+-----------+
|type_bien  |pieces|prix    |surface|etage|ville       |code_postal|
+-----------+------+--------+-------+-----+------------+-----------+
|appartement|2.0   |203431.0|21.0   |1    |venissieux  |69200      |
|appartement|2.0   |235000.0|38.9   |6    |lyon        |69007      |
|appartement|2.0   |454000.0|92.0   |1    |toussieu    |69780      |
|appartement|2.0   |118000.0|36.5   |1    |saint priest|69800      |
|appartement|2.0   |133473.0|76.1   |RDC  |tarare      |69170      |
+-----------+------+--------+-------+-----+------------+-----------+
only showing top 5 rows


In [18]:
total = df_union.count()

# Nombre de lignes distinctes
distinct = df_union.distinct().count()

print(f"Total lignes      : {total}")
print(f"Lignes distinctes : {distinct}")
print(f"Doublons          : {total - distinct}")

Total lignes      : 8130
Lignes distinctes : 8019
Doublons          : 111


In [19]:
df_doublons = (etreproprio.groupBy(df_union.columns)
                 .count()
                 .filter(F.col("count") > 1)
                 .orderBy(F.col("count").desc()))

df_doublons.show(truncate=False)
df_doublons.count()


+-----------+------+--------+-------+------+---------------------+-----------+-----+
|type_bien  |pieces|prix    |surface|etage |ville                |code_postal|count|
+-----------+------+--------+-------+------+---------------------+-----------+-----+
|appartement|2.0   |229000.0|46.0   |4     |lyon                 |69123      |3    |
|appartement|2.0   |149000.0|33.0   |1     |oullins pierre benite|69149      |2    |
|appartement|5.0   |645000.0|152.0  |1     |lyon                 |69123      |2    |
|maison     |5.0   |395000.0|126.0  |MAISON|irigny               |69100      |2    |
|maison     |4.0   |398000.0|105.0  |MAISON|decines charpieu     |69275      |2    |
|maison     |7.0   |305000.0|151.0  |MAISON|communay             |69272      |2    |
|appartement|2.0   |225000.0|49.0   |1     |villeurbanne         |69266      |2    |
|appartement|3.0   |260000.0|73.0   |7     |villeurbanne         |69266      |2    |
|appartement|3.0   |240000.0|64.0   |1     |lyon                 

17

In [20]:
df_doublons = (logimmo.groupBy(df_union.columns)
                 .count()
                 .filter(F.col("count") > 1)
                 .orderBy(F.col("count").desc()))

df_doublons.show(5,truncate=False)
df_doublons.count()

+-----------+------+------+-------+------+----------------+-----------+-----+
|type_bien  |pieces|prix  |surface|etage |ville           |code_postal|count|
+-----------+------+------+-------+------+----------------+-----------+-----+
|appartement|3     |229500|63.3   |4     |caluire et cuire|69300      |3    |
|appartement|2     |175000|44.3   |RDC   |rillieux la pape|69140      |3    |
|appartement|2     |274000|42.0   |1     |caluire et cuire|69300      |3    |
|appartement|3     |366000|55.0   |RDC   |caluire et cuire|69300      |3    |
|maison     |5     |638000|145.0  |MAISON|dommartin       |69380      |3    |
+-----------+------+------+-------+------+----------------+-----------+-----+
only showing top 5 rows


87

In [21]:
df_doublons = (df_union.groupBy(df_union.columns)
                 .count()
                 .filter(F.col("count") > 1)
                 .orderBy(F.col("count").desc()))

df_doublons.show(truncate=False)
df_doublons.count()

+-----------+------+--------+-------+------+--------------------+-----------+-----+
|type_bien  |pieces|prix    |surface|etage |ville               |code_postal|count|
+-----------+------+--------+-------+------+--------------------+-----------+-----+
|maison     |5.0   |638000.0|145.0  |MAISON|dommartin           |69380      |3    |
|appartement|2.0   |175000.0|44.3   |RDC   |rillieux la pape    |69140      |3    |
|appartement|5.0   |690000.0|105.0  |3     |lyon                |69009      |3    |
|appartement|3.0   |366000.0|55.0   |RDC   |caluire et cuire    |69300      |3    |
|appartement|3.0   |229500.0|63.3   |4     |caluire et cuire    |69300      |3    |
|appartement|2.0   |274000.0|42.0   |1     |caluire et cuire    |69300      |3    |
|appartement|2.0   |229000.0|46.0   |4     |lyon                |69123      |3    |
|appartement|3.0   |215000.0|61.0   |5     |saint priest        |69800      |2    |
|appartement|5.0   |549000.0|117.0  |3     |lyon                |69007      

104

In [22]:
df_clean = df_union.distinct()


df_doublons = (df_clean.groupBy(df_union.columns)
                 .count()
                 .filter(F.col("count") > 1)
                 .orderBy(F.col("count").desc()))

df_doublons.show(truncate=False)
df_doublons.count()

+---------+------+----+-------+-----+-----+-----------+-----+
|type_bien|pieces|prix|surface|etage|ville|code_postal|count|
+---------+------+----+-------+-----+-----+-----------+-----+
+---------+------+----+-------+-----+-----+-----------+-----+



0

In [24]:
df_clean.count()

8019

In [23]:
df_clean.toPandas().to_csv("bases_fusionnees.csv", index=False, encoding="utf-8-sig")